In [1]:
!pip install osmnx folium networkx -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.4/104.4 kB 2.4 MB/s eta 0:00:00


In [3]:
# @title Import Library
Import Library
import osmnx as ox
import networkx as nx
import folium
import numpy as np
import math

print(f"OSMnx version: {ox.__version__}")
print(f"NetworkX version: {nx.__version__}")

OSMnx version: 2.1.0
NetworkX version: 3.6.1


In [4]:
# @title Download Peta Jalan Medan
#Download graph jaringan jalan Kota Medan
G = ox.graph_from_place(
    "Medan, North Sumatra, Indonesia",
    network_type="drive" #hanya jalan yang bisa dilalui kendaraan
)

#Info dasar graph
print(f"Jumlah node (persimpangan): {G.number_of_nodes()}")
print(f"Jumlah edge (ruas jalan): {G.number_of_edges()}")

Jumlah node (persimpangan) : 42678
Jumlah edge (ruas jalan)   : 99720


In [5]:
# @title Definisi Titik Banjir & Posko Evakuasi

#TITIK BANJIR HISTORIS (berdasarkan berita 2020-2025)
#Koordinat: [latitude, longitude]
flood_zones = {
    "Kampung Lalang - Medan Sunggal": (-3.5789, 98.6432),
    "Brigjend Katamso - Medan Maimun": (-3.5953, 98.6789),
    "Gaperta Ujung - Medan Helvetia": (-3.5612, 98.6521),
    "Jalan Pancing - Medan Tembung": (-3.5701, 98.7234),
    "Medan Marelan - Kawasan Utara": (-3.5234, 98.6678),
    "Terminal Amplas - Medan Amplas": (-3.6312, 98.7012),
    "Sei Mati - Medan Maimun": (-3.5987, 98.6812),
    "Tanjung Mulia - Medan Deli": (-3.5478, 98.7123),
}

#POSKO EVAKUASI (kantor camat / rumah sakit / masjid besar)
evacuation_posts = {
    "Kantor Camat Medan Helvetia": (-3.5598, 98.6489),
    "RSU Adam Malik": (-3.5667, 98.6789),
    "Kantor Camat Medan Sunggal": (-3.5812, 98.6401),
    "Masjid Raya Al-Mashun": (-3.5756, 98.6803),
    "Kantor Camat Medan Maimun": (-3.5934, 98.6756),
}

print(f"Titik banjir historis : {len(flood_zones)} lokasi")
print(f"Posko evakuasi : {len(evacuation_posts)} lokasi")
print("\nTitik Banjir:")
for nama, koordinat in flood_zones.items():
    print(f"  - {nama}: {koordinat}")
print("\nPosko Evakuasi:")
for nama, koordinat in evacuation_posts.items():
    print(f"  - {nama}: {koordinat}")

Titik banjir historis : 8 lokasi
Posko evakuasi : 5 lokasi

Titik Banjir:
  - Kampung Lalang - Medan Sunggal: (-3.5789, 98.6432)
  - Brigjend Katamso - Medan Maimun: (-3.5953, 98.6789)
  - Gaperta Ujung - Medan Helvetia: (-3.5612, 98.6521)
  - Jalan Pancing - Medan Tembung: (-3.5701, 98.7234)
  - Medan Marelan - Kawasan Utara: (-3.5234, 98.6678)
  - Terminal Amplas - Medan Amplas: (-3.6312, 98.7012)
  - Sei Mati - Medan Maimun: (-3.5987, 98.6812)
  - Tanjung Mulia - Medan Deli: (-3.5478, 98.7123)

Posko Evakuasi:
  - Kantor Camat Medan Helvetia: (-3.5598, 98.6489)
  - RSU Adam Malik: (-3.5667, 98.6789)
  - Kantor Camat Medan Sunggal: (-3.5812, 98.6401)
  - Masjid Raya Al-Mashun: (-3.5756, 98.6803)
  - Kantor Camat Medan Maimun: (-3.5934, 98.6756)


In [6]:
# @title Snap Koordinat ke Node Graph

#Snap koordinat ke node terdekat di graph jalan Medan karena koordinat manual belum tentu tepat di persimpangan

#Snap titik banjir
flood_nodes = {}
for nama, (lat, lon) in flood_zones.items():
    node = ox.distance.nearest_nodes(G, lon, lat)
    flood_nodes[nama] = node

#Snap posko evakuasi
evac_nodes = {}
for nama, (lat, lon) in evacuation_posts.items():
    node = ox.distance.nearest_nodes(G, lon, lat)
    evac_nodes[nama] = node

print("Node ID titik banjir:")
for nama, node in flood_nodes.items():
    print(f"  - {nama}: node {node}")

print("\nNode ID posko evakuasi:")
for nama, node in evac_nodes.items():
    print(f"  - {nama}: node {node}")

Node ID titik banjir:
  - Kampung Lalang - Medan Sunggal: node 5344145890
  - Brigjend Katamso - Medan Maimun: node 5344145890
  - Gaperta Ujung - Medan Helvetia: node 5344145890
  - Jalan Pancing - Medan Tembung: node 5344145890
  - Medan Marelan - Kawasan Utara: node 5344145890
  - Terminal Amplas - Medan Amplas: node 5344145890
  - Sei Mati - Medan Maimun: node 5344145890
  - Tanjung Mulia - Medan Deli: node 5344145890

Node ID posko evakuasi:
  - Kantor Camat Medan Helvetia: node 5344145890
  - RSU Adam Malik: node 5344145890
  - Kantor Camat Medan Sunggal: node 5344145890
  - Masjid Raya Al-Mashun: node 5344145890
  - Kantor Camat Medan Maimun: node 5344145890


In [7]:
# @title Cek Batas Wilayah Graph
#Cek batas koordinat graph yang berhasil diunduh
nodes_gdf = ox.graph_to_gdfs(G, edges=False)

lat_min = nodes_gdf['y'].min()
lat_max = nodes_gdf['y'].max()
lon_min = nodes_gdf['x'].min()
lon_max = nodes_gdf['x'].max()
lat_center = nodes_gdf['y'].mean()
lon_center = nodes_gdf['x'].mean()

print(f"Batas wilayah graph:")
print(f"  Latitude  : {lat_min:.6f} s/d {lat_max:.6f}")
print(f"  Longitude : {lon_min:.6f} s/d {lon_max:.6f}")
print(f"  Titik tengah: ({lat_center:.6f}, {lon_center:.6f})")

Batas wilayah graph:
  Latitude  : 3.488142 s/d 3.787459
  Longitude : 98.594767 s/d 98.745528
  Titik tengah: (3.608108, 98.667513)


In [8]:
# @title Koreksi Koordinat & Snap Ulang
#KOORDINAT DIKOREKSI (latitude positif, sesuai batas graph)
flood_zones = {
    "Kampung Lalang - Medan Sunggal": (3.5789, 98.6432),
    "Brigjend Katamso - Medan Maimun": (3.5953, 98.6789),
    "Gaperta Ujung - Medan Helvetia": (3.5612, 98.6521),
    "Jalan Pancing - Medan Tembung": (3.5701, 98.7234),
    "Medan Marelan - Kawasan Utara": (3.5234, 98.6678),
    "Terminal Amplas - Medan Amplas": (3.6312, 98.7012),
    "Sei Mati - Medan Maimun": (3.5987, 98.6812),
    "Tanjung Mulia - Medan Deli": (3.5478, 98.7123),
}

evacuation_posts = {
    "Kantor Camat Medan Helvetia": (3.5598, 98.6489),
    "RSU Adam Malik": (3.5667, 98.6789),
    "Kantor Camat Medan Sunggal": (3.5812, 98.6401),
    "Masjid Raya Al-Mashun": (3.5756, 98.6803),
    "Kantor Camat Medan Maimun": (3.5934, 98.6756),
}

#Snap ulang ke node graph
flood_nodes = {}
for nama, (lat, lon) in flood_zones.items():
    node = ox.distance.nearest_nodes(G, lon, lat)
    flood_nodes[nama] = node

evac_nodes = {}
for nama, (lat, lon) in evacuation_posts.items():
    node = ox.distance.nearest_nodes(G, lon, lat)
    evac_nodes[nama] = node

print("Node ID titik banjir:")
for nama, node in flood_nodes.items():
    print(f"  - {nama}: node {node}")

print("\nNode ID posko evakuasi:")
for nama, node in evac_nodes.items():
    print(f"  - {nama}: node {node}")

Node ID titik banjir:
  - Kampung Lalang - Medan Sunggal: node 687151156
  - Brigjend Katamso - Medan Maimun: node 5474533661
  - Gaperta Ujung - Medan Helvetia: node 5391642125
  - Jalan Pancing - Medan Tembung: node 5358544331
  - Medan Marelan - Kawasan Utara: node 5358381374
  - Terminal Amplas - Medan Amplas: node 1303402335
  - Sei Mati - Medan Maimun: node 6817061190
  - Tanjung Mulia - Medan Deli: node 6596609342

Node ID posko evakuasi:
  - Kantor Camat Medan Helvetia: node 5391694465
  - RSU Adam Malik: node 4992678982
  - Kantor Camat Medan Sunggal: node 5490200380
  - Masjid Raya Al-Mashun: node 3463729512
  - Kantor Camat Medan Maimun: node 1724558971


In [9]:
# @title Implementasi Algoritma A*
#FUNGSI HEURISTIK: Haversine Distance
#Menghitung jarak sesungguhnya antara dua koordinat GPS

def haversine(u, v, G):
    lat1 = math.radians(G.nodes[u]['y'])
    lon1 = math.radians(G.nodes[u]['x'])
    lat2 = math.radians(G.nodes[v]['y'])
    lon2 = math.radians(G.nodes[v]['x'])

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = math.sin(dlat/2)**2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon/2)**2
    c = 2 * math.asin(math.sqrt(a))
    r = 6371000  #radius bumi dalam meter
    return c * r

#FUNGSI A* PATHFINDING
#Mencari rute terpendek dari titik korban ke posko evakuasi
#blocked_edges: set of (u, v) yang tidak bisa dilalui (banjir)

def astar_evacuation(G, start_node, goal_node, blocked_edges=set()):
    #Buat graph sementara dengan edge banjir dihapus
    G_temp = G.copy()
    G_temp.remove_edges_from(blocked_edges)

    try:
        path = nx.astar_path(
            G_temp,
            source=start_node,
            target=goal_node,
            heuristic=lambda u, v: haversine(u, v, G),
            weight='length'
        )

        #Hitung total jarak rute
        total_distance = sum(
            G_temp[path[i]][path[i+1]][0]['length']
            for i in range(len(path)-1)
        )

        return path, total_distance

    except nx.NetworkXNoPath:
        return None, None

print("   - haversine()         : fungsi heuristik jarak GPS")
print("   - astar_evacuation()  : fungsi pencarian rute optimal")

   - haversine()         : fungsi heuristik jarak GPS
   - astar_evacuation()  : fungsi pencarian rute optimal


In [10]:
# @title Simulasi Skenario & Jalankan A*
#SIMULASI TITIK JALAN TERGENANG (berdasarkan zona banjir)
#Ambil semua edge dalam radius 300m dari titik banjir
def get_flooded_edges(G, flood_zones, radius_meter=300):
    flooded = set()
    nodes_gdf = ox.graph_to_gdfs(G, edges=False)

    for nama, (lat, lon) in flood_zones.items():
        #Cari semua node dalam radius dari titik banjir
        for node, data in G.nodes(data=True):
            dist = haversine_coord(lat, lon, data['y'], data['x'])
            if dist <= radius_meter:
                #Tandai semua edge yang terhubung ke node ini
                for u, v, k in G.edges(node, keys=True):
                    flooded.add((u, v))
                    flooded.add((v, u))
    return flooded

def haversine_coord(lat1, lon1, lat2, lon2):
    r = 6371000
    p = math.pi / 180
    a = (math.sin((lat2-lat1)*p/2)**2 +
         math.cos(lat1*p) * math.cos(lat2*p) *
         math.sin((lon2-lon1)*p/2)**2)
    return 2 * r * math.asin(math.sqrt(a))

#SKENARIO BANJIR
scenarios = {
    "Skenario A - Banjir Ringan (3 titik)": dict(list(flood_zones.items())[:3]),
    "Skenario B - Banjir Sedang (5 titik)": dict(list(flood_zones.items())[:5]),
    "Skenario C - Banjir Parah (8 titik)":  flood_zones,
}

#JALANKAN A* - Dari setiap titik banjir ke posko TERDEKAT
results = {}  # simpan hasil untuk visualisasi

for skenario_nama, active_floods in scenarios.items():
    print(f"\n{'='*55}")
    print(f" {skenario_nama}")
    print(f"{'='*55}")

    blocked = get_flooded_edges(G, active_floods)
    print(f"   Edge terblokir: {len(blocked)} ruas jalan\n")

    skenario_results = []

    for banjir_nama, banjir_node in flood_nodes.items():
        best_path = None
        best_dist = float('inf')
        best_posko = None

        #Cari posko terdekat yang bisa dicapai
        for posko_nama, posko_node in evac_nodes.items():
            path, dist = astar_evacuation(G, banjir_node, posko_node, blocked)
            if dist is not None and dist < best_dist:
                best_dist = dist
                best_path = path
                best_posko = posko_nama

        if best_path:
            print(f"   {banjir_nama}")
            print(f"     ===> Posko: {best_posko}")
            print(f"     ===> Jarak: {best_dist/1000:.2f} km | {len(best_path)} node")
        else:
            print(f"   {banjir_nama} — Tidak ada rute.")

        skenario_results.append({
            "asal": banjir_nama,
            "posko": best_posko,
            "path": best_path,
            "jarak_km": best_dist/1000 if best_dist != float('inf') else None
        })

    results[skenario_nama] = skenario_results

print(f"\n\n Semua skenario selesai diproses.")


 Skenario A - Banjir Ringan (3 titik)
   Edge terblokir: 390 ruas jalan

   Kampung Lalang - Medan Sunggal — Tidak ada rute.
   Brigjend Katamso - Medan Maimun — Tidak ada rute.
   Gaperta Ujung - Medan Helvetia — Tidak ada rute.
   Jalan Pancing - Medan Tembung
     ===> Posko: Masjid Raya Al-Mashun
     ===> Jarak: 5.52 km | 79 node
   Medan Marelan - Kawasan Utara
     ===> Posko: RSU Adam Malik
     ===> Jarak: 7.80 km | 112 node
   Terminal Amplas - Medan Amplas
     ===> Posko: Masjid Raya Al-Mashun
     ===> Jarak: 9.03 km | 140 node
   Sei Mati - Medan Maimun
     ===> Posko: Masjid Raya Al-Mashun
     ===> Jarak: 3.92 km | 57 node
   Tanjung Mulia - Medan Deli
     ===> Posko: Masjid Raya Al-Mashun
     ===> Jarak: 5.97 km | 92 node

 Skenario B - Banjir Sedang (5 titik)
   Edge terblokir: 693 ruas jalan

   Kampung Lalang - Medan Sunggal — Tidak ada rute.
   Brigjend Katamso - Medan Maimun — Tidak ada rute.
   Gaperta Ujung - Medan Helvetia — Tidak ada rute.
   Jalan Pancing

In [11]:
# @title Perbaikan Logika Blocked Edges

#PERBAIKAN: Blocked edges hanya jalan DALAM zona banjir bukan jalan yang keluar dari zona banjir
#Solusi: hanya blokir edge dimana KEDUA ujungnya di zona banjir

def get_flooded_edges_v2(G, flood_zones, radius_meter=300):
    #Kumpulkan semua node yang berada dalam zona banjir
    flooded_nodes = set()
    for nama, (lat, lon) in flood_zones.items():
        for node, data in G.nodes(data=True):
            dist = haversine_coord(lat, lon, data['y'], data['x'])
            if dist <= radius_meter:
                flooded_nodes.add(node)

    #Hanya blokir edge dimana KEDUA ujung node ada di zona banjir
    flooded_edges = set()
    for u, v, k in G.edges(keys=True):
        if u in flooded_nodes and v in flooded_nodes:
            flooded_edges.add((u, v))
            flooded_edges.add((v, u))

    return flooded_edges, flooded_nodes

#JALANKAN ULANG SEMUA SKENARIO
results = {}

for skenario_nama, active_floods in scenarios.items():
    print(f"\n{'='*55}")
    print(f" {skenario_nama}")
    print(f"{'='*55}")

    blocked, flooded_nodes = get_flooded_edges_v2(G, active_floods)
    print(f"   Node tergenang : {len(flooded_nodes)} persimpangan")
    print(f"   Edge terblokir : {len(blocked)} ruas jalan\n")

    skenario_results = []

    for banjir_nama, banjir_node in flood_nodes.items():
        best_path = None
        best_dist = float('inf')
        best_posko = None

        for posko_nama, posko_node in evac_nodes.items():
            path, dist = astar_evacuation(G, banjir_node, posko_node, blocked)
            if dist is not None and dist < best_dist:
                best_dist = dist
                best_path = path
                best_posko = posko_nama

        if best_path:
            print(f"   {banjir_nama}")
            print(f"     ===> Posko : {best_posko}")
            print(f"     ===> Jarak : {best_dist/1000:.2f} km | {len(best_path)} node")
        else:
            print(f"   {banjir_nama} — Tidak ada rute.")

        skenario_results.append({
            "asal": banjir_nama,
            "posko": best_posko,
            "path": best_path,
            "jarak_km": best_dist/1000 if best_dist != float('inf') else None
        })

    results[skenario_nama] = skenario_results

print(f"\n Semua skenario selesai diproses.")


 Skenario A - Banjir Ringan (3 titik)
   Node tergenang : 148 persimpangan
   Edge terblokir : 302 ruas jalan

   Kampung Lalang - Medan Sunggal — Tidak ada rute.
   Brigjend Katamso - Medan Maimun — Tidak ada rute.
   Gaperta Ujung - Medan Helvetia — Tidak ada rute.
   Jalan Pancing - Medan Tembung
     ===> Posko : Masjid Raya Al-Mashun
     ===> Jarak : 5.52 km | 79 node
   Medan Marelan - Kawasan Utara
     ===> Posko : Kantor Camat Medan Helvetia
     ===> Jarak : 7.12 km | 82 node
   Terminal Amplas - Medan Amplas
     ===> Posko : Masjid Raya Al-Mashun
     ===> Jarak : 9.03 km | 140 node
   Sei Mati - Medan Maimun
     ===> Posko : Masjid Raya Al-Mashun
     ===> Jarak : 3.92 km | 57 node
   Tanjung Mulia - Medan Deli
     ===> Posko : Masjid Raya Al-Mashun
     ===> Jarak : 5.97 km | 92 node

 Skenario B - Banjir Sedang (5 titik)
   Node tergenang : 268 persimpangan
   Edge terblokir : 529 ruas jalan

   Kampung Lalang - Medan Sunggal — Tidak ada rute.
   Brigjend Katamso - M

In [12]:
# @title Diagnosa Node Terisolasi

#Cek komponen terbesar graph
largest_component = max(nx.weakly_connected_components(G), key=len)
print(f"Total node di graph : {G.number_of_nodes()}")
print(f"Node di komponen terbesar : {len(largest_component)}")
print(f"Node di luar komponen utama : {G.number_of_nodes() - len(largest_component)}")

#Cek apakah node banjir & posko ada di komponen terbesar
print("\nStatus node titik banjir:")
for nama, node in flood_nodes.items():
    status = " terhubung" if node in largest_component else " TERISOLASI"
    print(f"  {status} | {nama}: node {node}")

print("\nStatus node posko evakuasi:")
for nama, node in evac_nodes.items():
    status = " terhubung" if node in largest_component else " TERISOLASI"
    print(f"  {status} | {nama}: node {node}")

Total node di graph : 42678
Node di komponen terbesar : 42678
Node di luar komponen utama : 0

Status node titik banjir:
   terhubung | Kampung Lalang - Medan Sunggal: node 687151156
   terhubung | Brigjend Katamso - Medan Maimun: node 5474533661
   terhubung | Gaperta Ujung - Medan Helvetia: node 5391642125
   terhubung | Jalan Pancing - Medan Tembung: node 5358544331
   terhubung | Medan Marelan - Kawasan Utara: node 5358381374
   terhubung | Terminal Amplas - Medan Amplas: node 1303402335
   terhubung | Sei Mati - Medan Maimun: node 6817061190
   terhubung | Tanjung Mulia - Medan Deli: node 6596609342

Status node posko evakuasi:
   terhubung | Kantor Camat Medan Helvetia: node 5391694465
   terhubung | RSU Adam Malik: node 4992678982
   terhubung | Kantor Camat Medan Sunggal: node 5490200380
   terhubung | Masjid Raya Al-Mashun: node 3463729512
   terhubung | Kantor Camat Medan Maimun: node 1724558971


In [13]:
# @title Diagnosa Node Start vs Flooded Nodes
#Cek apakah node titik banjir masuk ke dalam flooded_nodes
#Gunakan Skenario A dulu (3 titik banjir)
active_floods_A = dict(list(flood_zones.items())[:3])
blocked_A, flooded_nodes_A = get_flooded_edges_v2(G, active_floods_A)

print("Skenario A — Cek node start vs flooded_nodes:\n")
for nama, node in flood_nodes.items():
    in_flood = "  ADA dalam zona banjir" if node in flooded_nodes_A else " di luar zona banjir"
    #Cek berapa edge keluar yang masih tersedia
    edges_out = [(u,v) for u,v in G.out_edges(node) if (u,v) not in blocked_A]
    print(f"  {in_flood} | {nama}")
    print(f"    Node: {node} | Edge keluar tersisa: {len(edges_out)}")

Skenario A — Cek node start vs flooded_nodes:

    ADA dalam zona banjir | Kampung Lalang - Medan Sunggal
    Node: 687151156 | Edge keluar tersisa: 0
    ADA dalam zona banjir | Brigjend Katamso - Medan Maimun
    Node: 5474533661 | Edge keluar tersisa: 0
    ADA dalam zona banjir | Gaperta Ujung - Medan Helvetia
    Node: 5391642125 | Edge keluar tersisa: 0
   di luar zona banjir | Jalan Pancing - Medan Tembung
    Node: 5358544331 | Edge keluar tersisa: 1
   di luar zona banjir | Medan Marelan - Kawasan Utara
    Node: 5358381374 | Edge keluar tersisa: 3
   di luar zona banjir | Terminal Amplas - Medan Amplas
    Node: 1303402335 | Edge keluar tersisa: 1
   di luar zona banjir | Sei Mati - Medan Maimun
    Node: 6817061190 | Edge keluar tersisa: 2
   di luar zona banjir | Tanjung Mulia - Medan Deli
    Node: 6596609342 | Edge keluar tersisa: 3


In [14]:
# @title Perbaikan Final: Cari Node Keluar Terdekat
#Fungsi: cari node terdekat di luar zona banjir
#Simulasi: korban berjalan kaki ke tepi zona banjir dulu
def find_nearest_safe_node(G, start_node, flooded_nodes, blocked_edges):
    #Jika start node aman, langsung return
    if start_node not in flooded_nodes:
        return start_node

    #BFS dari start_node, cari node pertama yang punya edge keluar bebas
    visited = set()
    queue = [start_node]

    while queue:
        current = queue.pop(0)
        if current in visited:
            continue
        visited.add(current)

        #Cek apakah node ini punya edge keluar yang tidak terblokir
        edges_out = [(u,v) for u,v in G.out_edges(current)
                     if (u,v) not in blocked_edges and v not in flooded_nodes]

        if len(edges_out) > 0 and current != start_node:
            return current  #node aman ditemukan

        #Tambah tetangga ke queue (meski masih di zona banjir)
        for _, neighbor in G.out_edges(current):
            if neighbor not in visited:
                queue.append(neighbor)

    return None  #tidak ada jalan keluar sama sekali

#JALANKAN ULANG SEMUA SKENARIO DENGAN PERBAIKAN
results = {}

for skenario_nama, active_floods in scenarios.items():
    print(f"\n{'='*55}")
    print(f" {skenario_nama}")
    print(f"{'='*55}")

    blocked, flooded_nodes = get_flooded_edges_v2(G, active_floods)
    print(f"   Node tergenang : {len(flooded_nodes)} persimpangan")
    print(f"   Edge terblokir : {len(blocked)} ruas jalan\n")

    skenario_results = []

    for banjir_nama, banjir_node in flood_nodes.items():
        #Cari node aman terdekat jika start node di zona banjir
        safe_start = find_nearest_safe_node(G, banjir_node, flooded_nodes, blocked)

        if safe_start is None:
            print(f"   {banjir_nama} - Terkepung total, tidak ada jalan keluar!")
            skenario_results.append({
                "asal": banjir_nama, "posko": None,
                "path": None, "jarak_km": None
            })
            continue

        best_path = None
        best_dist = float('inf')
        best_posko = None

        for posko_nama, posko_node in evac_nodes.items():
            path, dist = astar_evacuation(G, safe_start, posko_node, blocked)
            if dist is not None and dist < best_dist:
                best_dist = dist
                best_path = path
                best_posko = posko_nama

        terjebak = " (keluar zona dulu)" if safe_start != banjir_node else ""

        if best_path:
            print(f"   {banjir_nama}{terjebak}")
            print(f"     ===> Posko : {best_posko}")
            print(f"     ===> Jarak : {best_dist/1000:.2f} km | {len(best_path)} node")
        else:
            print(f"   {banjir_nama}  Tidak ada rute ke posko manapun.")

        skenario_results.append({
            "asal": banjir_nama,
            "posko": best_posko,
            "path": best_path,
            "jarak_km": best_dist/1000 if best_dist != float('inf') else None
        })

    results[skenario_nama] = skenario_results

print(f"\n Semua skenario selesai diproses.")


 Skenario A - Banjir Ringan (3 titik)
   Node tergenang : 148 persimpangan
   Edge terblokir : 302 ruas jalan

   Kampung Lalang - Medan Sunggal (keluar zona dulu)
     ===> Posko : Kantor Camat Medan Sunggal
     ===> Jarak : 0.80 km | 13 node
   Brigjend Katamso - Medan Maimun  Tidak ada rute ke posko manapun.
   Gaperta Ujung - Medan Helvetia (keluar zona dulu)
     ===> Posko : Kantor Camat Medan Helvetia
     ===> Jarak : 1.20 km | 12 node
   Jalan Pancing - Medan Tembung
     ===> Posko : Masjid Raya Al-Mashun
     ===> Jarak : 5.52 km | 79 node
   Medan Marelan - Kawasan Utara
     ===> Posko : Kantor Camat Medan Helvetia
     ===> Jarak : 7.12 km | 82 node
   Terminal Amplas - Medan Amplas
     ===> Posko : Masjid Raya Al-Mashun
     ===> Jarak : 9.03 km | 140 node
   Sei Mati - Medan Maimun
     ===> Posko : Masjid Raya Al-Mashun
     ===> Jarak : 3.92 km | 57 node
   Tanjung Mulia - Medan Deli
     ===> Posko : Masjid Raya Al-Mashun
     ===> Jarak : 5.97 km | 92 node

 Sken

In [15]:
# @title Visualisasi Peta Folium
#VISUALISASI PETA INTERAKTIF
#Tampilkan Skenario C (paling parah) sebagai visualisasi utama

#Ambil data Skenario C
skenario_viz = "Skenario C - Banjir Parah (8 titik)"
active_floods_viz = flood_zones
blocked_viz, flooded_nodes_viz = get_flooded_edges_v2(G, active_floods_viz)

#Buat peta centered di Medan
m = folium.Map(
    location=[3.608108, 98.667513],
    zoom_start=13,
    tiles="OpenStreetMap"
)

#1. Gambar zona banjir (lingkaran merah transparan)
for nama, (lat, lon) in active_floods_viz.items():
    folium.Circle(
        location=[lat, lon],
        radius=300,
        color="red",
        fill=True,
        fill_opacity=0.25,
        tooltip=f" Zona Banjir: {nama}"
    ).add_to(m)

#2. Marker titik banjir (ikon merah)
for nama, (lat, lon) in active_floods_viz.items():
    folium.Marker(
        location=[lat, lon],
        icon=folium.Icon(color="red", icon="tint", prefix="fa"),
        tooltip=f" {nama}"
    ).add_to(m)

#3. Marker posko evakuasi (ikon hijau)
for nama, (lat, lon) in evacuation_posts.items():
    folium.Marker(
        location=[lat, lon],
        icon=folium.Icon(color="green", icon="plus-sign"),
        tooltip=f" Posko: {nama}"
    ).add_to(m)

#4. Gambar rute A* untuk setiap titik
colors = ["blue","purple","orange","darkblue","cadetblue",
          "darkgreen","black","darkred"]

for i, item in enumerate(results[skenario_viz]):
    if item["path"] is None:
        continue

    #Ambil koordinat setiap node di path
    route_coords = [
        (G.nodes[node]["y"], G.nodes[node]["x"])
        for node in item["path"]
    ]

    folium.PolyLine(
        locations=route_coords,
        color=colors[i % len(colors)],
        weight=4,
        opacity=0.8,
        tooltip=f" Rute: {item['asal']} → {item['posko']} ({item['jarak_km']:.2f} km)"
    ).add_to(m)

#5. Legend manual
legend_html = """
<div style="position: fixed; bottom: 30px; left: 30px; z-index: 1000;
     background-color: white; padding: 12px; border-radius: 8px;
     border: 2px solid grey; font-size: 13px;">
<b> Legenda</b><br>
<span style="color:red">●</span> Zona Banjir<br>
<span style="color:green">●</span> Posko Evakuasi<br>
<span style="color:blue">─</span> Rute Evakuasi A*<br>
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

#Simpan peta
m.save("peta_evakuasi_medan.html")
print(" Peta berhasil disimpan: peta_evakuasi_medan.html")
print("   Buka file tersebut di browser untuk melihat peta interaktif")

 Peta berhasil disimpan: peta_evakuasi_medan.html
   Buka file tersebut di browser untuk melihat peta interaktif


In [16]:
# @title Cara 1 Tampilkan Langsung di Colab
from IPython.display import display, HTML

with open("peta_evakuasi_medan.html", "r") as f:
    html_content = f.read()

display(HTML(html_content))

In [17]:
# @title Cara 2 Download file HTML
from google.colab import files
files.download("peta_evakuasi_medan.html")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [18]:
# @title Tabel Ringkasan Hasil

#TABEL PERBANDINGAN HASIL SEMUA SKENARIO
print("=" * 75)
print(f"{'RINGKASAN HASIL SIMULASI RUTE EVAKUASI BANJIR KOTA MEDAN':^75}")
print(f"{'Algoritma A* (A-Star) Pathfinding':^75}")
print("=" * 75)

for skenario_nama, skenario_results in results.items():
    berhasil = [r for r in skenario_results if r["path"] is not None]
    gagal    = [r for r in skenario_results if r["path"] is None]

    jarak_list = [r["jarak_km"] for r in berhasil]
    avg_jarak  = sum(jarak_list) / len(jarak_list) if jarak_list else 0
    max_jarak  = max(jarak_list) if jarak_list else 0
    min_jarak  = min(jarak_list) if jarak_list else 0

    print(f"\n {skenario_nama}")
    print(f"   Rute berhasil  : {len(berhasil)}/8 titik")
    print(f"   Terkepung total: {len(gagal)}/8 titik")
    print(f"   Jarak rata-rata: {avg_jarak:.2f} km")
    print(f"   Jarak terpendek: {min_jarak:.2f} km")
    print(f"   Jarak terjauh  : {max_jarak:.2f} km")
    print(f"   {'─'*55}")
    print(f"   {'Lokasi Banjir':<35} {'Posko Tujuan':<25} {'Jarak':>6}")
    print(f"   {'─'*55}")
    for r in skenario_results:
        if r["path"]:
            asal   = r["asal"][:33]
            posko  = r["posko"][:23] if r["posko"] else "-"
            jarak  = f"{r['jarak_km']:.2f} km"
            print(f"   {asal:<35} {posko:<25} {jarak:>8}")
        else:
            asal = r["asal"][:33]
            print(f"   {asal:<35} {' Terkepung total':<25} {'---':>8}")

print("\n" + "=" * 75)
print("Kesimpulan:")
print("  - Masjid Raya Al-Mashun menjadi posko paling banyak dituju")
print("  - Semakin parah banjir, semakin banyak titik terkepung")
print("  - Brigjend Katamso terkepung total di semua skenario")
print("  - Algoritma A* berhasil menemukan rute alternatif")
print("    meski jalan utama terblokir banjir")
print("=" * 75)

         RINGKASAN HASIL SIMULASI RUTE EVAKUASI BANJIR KOTA MEDAN          
                     Algoritma A* (A-Star) Pathfinding                     

 Skenario A - Banjir Ringan (3 titik)
   Rute berhasil  : 7/8 titik
   Terkepung total: 1/8 titik
   Jarak rata-rata: 4.79 km
   Jarak terpendek: 0.80 km
   Jarak terjauh  : 9.03 km
   ───────────────────────────────────────────────────────
   Lokasi Banjir                       Posko Tujuan               Jarak
   ───────────────────────────────────────────────────────
   Kampung Lalang - Medan Sunggal      Kantor Camat Medan Sung    0.80 km
   Brigjend Katamso - Medan Maimun      Terkepung total               ---
   Gaperta Ujung - Medan Helvetia      Kantor Camat Medan Helv    1.20 km
   Jalan Pancing - Medan Tembung       Masjid Raya Al-Mashun      5.52 km
   Medan Marelan - Kawasan Utara       Kantor Camat Medan Helv    7.12 km
   Terminal Amplas - Medan Amplas      Masjid Raya Al-Mashun      9.03 km
   Sei Mati - Medan Maimun     

In [19]:
# @title Simpan Hasil & Download Semua File
import json
from google.colab import files

#1. Simpan hasil ringkasan ke file teks
summary_text = """===========================================================================
         RINGKASAN HASIL SIMULASI RUTE EVAKUASI BANJIR KOTA MEDAN
                     Algoritma A* (A-Star) Pathfinding
===========================================================================
Dibuat oleh : MINDBYTE
Dataset     : OpenStreetMap (OSMnx) + Simulasi Titik Banjir Historis Medan
Algoritma   : A* dengan heuristik Haversine Distance
Skenario    : A (3 titik), B (5 titik), C (8 titik banjir)
===========================================================================
"""

for skenario_nama, skenario_results in results.items():
    berhasil  = [r for r in skenario_results if r["path"] is not None]
    gagal     = [r for r in skenario_results if r["path"] is None]
    jarak_list = [r["jarak_km"] for r in berhasil]
    avg_jarak  = sum(jarak_list) / len(jarak_list) if jarak_list else 0

    summary_text += f"\n{skenario_nama}\n"
    summary_text += f"  Berhasil: {len(berhasil)}/8 | Terkepung: {len(gagal)}/8 | Rata-rata jarak: {avg_jarak:.2f} km\n"
    for r in skenario_results:
        if r["path"]:
            summary_text += f"   {r['asal'][:35]:<35} → {r['posko'][:25]:<25} {r['jarak_km']:.2f} km\n"
        else:
            summary_text += f"   {r['asal'][:35]:<35} → Terkepung total\n"

summary_text += "\n===========================================================================\n"

with open("hasil_simulasi.txt", "w") as f:
    f.write(summary_text)
print(" hasil_simulasi.txt tersimpan")

#2. Simpan data hasil ke JSON (untuk keperluan lanjutan)
results_json = {}
for skenario_nama, skenario_results in results.items():
    results_json[skenario_nama] = [
        {
            "asal": r["asal"],
            "posko": r["posko"],
            "jarak_km": r["jarak_km"],
            "jumlah_node": len(r["path"]) if r["path"] else 0,
            "status": "berhasil" if r["path"] else "terkepung"
        }
        for r in skenario_results
    ]

with open("hasil_simulasi.json", "w") as f:
    json.dump(results_json, f, indent=2, ensure_ascii=False)
print(" hasil_simulasi.json tersimpan")

#3. Download semua file sekaligus
files.download("peta_evakuasi_medan.html")
files.download("hasil_simulasi.txt")
files.download("hasil_simulasi.json")
print("\n Semua file berhasil didownload:")
print("   peta_evakuasi_medan.html  ===> Peta interaktif")
print("   hasil_simulasi.txt        ===> Ringkasan teks")
print("   hasil_simulasi.json       ===> Data JSON")

 hasil_simulasi.txt tersimpan
 hasil_simulasi.json tersimpan


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


 Semua file berhasil didownload:
   peta_evakuasi_medan.html  ===> Peta interaktif
   hasil_simulasi.txt        ===> Ringkasan teks
   hasil_simulasi.json       ===> Data JSON
